# Assignment — Part 4: Data Visualization & Machine Learning
**Theme:** Student Performance Analysis & Prediction  
**Total Marks:** 25 (+2 bonus)  

This notebook covers:
1. Data Exploration with Pandas
2. Data Visualization with Matplotlib (5 plots)
3. Data Visualization with Seaborn (2 plots)
4. Machine Learning with scikit-learn (Logistic Regression)

## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Display plots inline in the notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
print('All libraries loaded successfully.')

---
## Task 1 — Data Exploration with Pandas  *(5 marks)*

In [ ]:
# Load the dataset
df = pd.read_csv('students.csv')

# 1. First 3 rows
print('=== First 3 rows ===')
print(df.head(3).to_string(index=False))

In [ ]:
# 2. Shape and data types
print(f'Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print()
print('Data types:')
print(df.dtypes)

In [ ]:
# 3. Summary statistics for numeric columns
print('=== Descriptive Statistics ===')
print(df.describe().round(2).to_string())

In [ ]:
# 4. Count of passed vs failed
counts = df['passed'].value_counts()
print('=== Pass / Fail Counts ===')
print(f"  Passed : {counts.get(1, 0)}")
print(f"  Failed : {counts.get(0, 0)}")

In [ ]:
# 5. Average score per subject — separately for passing and failing students
subject_cols = ['math', 'science', 'english', 'history', 'pe']

pass_avg = df[df['passed'] == 1][subject_cols].mean()
fail_avg = df[df['passed'] == 0][subject_cols].mean()

comparison = pd.DataFrame({'Pass Avg': pass_avg.round(2),
                           'Fail Avg': fail_avg.round(2)})
print('=== Average Score per Subject: Pass vs Fail ===')
print(comparison.to_string())

In [ ]:
# 6. Student with highest overall average across all 5 subjects
df_temp = df.copy()
df_temp['overall_avg'] = df_temp[subject_cols].mean(axis=1)
top_student = df_temp.loc[df_temp['overall_avg'].idxmax()]
print('=== Top Student ===')
print(f"  Name    : {top_student['name']}")
print(f"  Overall Average : {top_student['overall_avg']:.2f}")

---
## Task 2 — Data Visualization with Matplotlib  *(8 marks)*

Each plot is saved as a `.png` file **and** displayed inline.

In [ ]:
# Prepare avg_score column used in several plots
subject_cols = ['math', 'science', 'english', 'history', 'pe']
df['avg_score'] = df[subject_cols].mean(axis=1)

### Plot 1 — Bar Chart: Average Score per Subject

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
avgs = df[subject_cols].mean()
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
bars = ax.bar(avgs.index, avgs.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('Average Score per Subject (All Students)', fontsize=14, fontweight='bold')
ax.set_xlabel('Subject')
ax.set_ylabel('Average Score')
ax.set_ylim(0, 100)
for bar, val in zip(bars, avgs.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot1_bar_avg_subject.png', dpi=120)
plt.show()
print('Saved: plot1_bar_avg_subject.png')

### Plot 2 — Histogram: Distribution of Math Scores

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['math'], bins=5, color='#4C72B0', edgecolor='white', linewidth=0.8, alpha=0.85)
mean_math = df['math'].mean()
ax.axvline(mean_math, color='crimson', linestyle='--', linewidth=2,
           label=f'Mean: {mean_math:.1f}')
ax.set_title('Distribution of Math Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('Math Score')
ax.set_ylabel('Number of Students')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot2_hist_math.png', dpi=120)
plt.show()
print('Saved: plot2_hist_math.png')

### Plot 3 — Scatter Plot: Study Hours vs Average Score

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pass_df = df[df['passed'] == 1]
fail_df = df[df['passed'] == 0]
ax.scatter(pass_df['study_hours_per_day'], pass_df['avg_score'],
           color='#2ecc71', label='Pass', s=80, edgecolors='white', linewidth=0.5)
ax.scatter(fail_df['study_hours_per_day'], fail_df['avg_score'],
           color='#e74c3c', label='Fail', s=80, marker='X', edgecolors='white', linewidth=0.5)
ax.set_title('Study Hours per Day vs Average Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Study Hours per Day')
ax.set_ylabel('Average Score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot3_scatter_study_avg.png', dpi=120)
plt.show()
print('Saved: plot3_scatter_study_avg.png')

### Plot 4 — Box Plot: Attendance % for Pass vs Fail

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
pass_att = df[df['passed'] == 1]['attendance_pct'].tolist()
fail_att = df[df['passed'] == 0]['attendance_pct'].tolist()
bp = ax.boxplot([pass_att, fail_att], tick_labels=['Pass', 'Fail'],
                patch_artist=True,
                boxprops=dict(facecolor='#AED6F1'),
                medianprops=dict(color='navy', linewidth=2))
bp['boxes'][1].set_facecolor('#F1948A')
ax.set_title('Attendance % Distribution: Pass vs Fail', fontsize=14, fontweight='bold')
ax.set_xlabel('Result')
ax.set_ylabel('Attendance %')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot4_box_attendance.png', dpi=120)
plt.show()
print('Saved: plot4_box_attendance.png')

### Plot 5 — Line Plot: Math & Science Scores per Student

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df))
ax.plot(x, df['math'],    marker='o', linestyle='-',  label='Math',
        color='#2980B9', linewidth=2)
ax.plot(x, df['science'], marker='s', linestyle='--', label='Science',
        color='#E67E22', linewidth=2)
ax.set_xticks(list(x))
ax.set_xticklabels(df['name'], rotation=45, ha='right')
ax.set_title('Math vs Science Scores per Student', fontsize=14, fontweight='bold')
ax.set_xlabel('Student')
ax.set_ylabel('Score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot5_line_math_science.png', dpi=120)
plt.show()
print('Saved: plot5_line_math_science.png')

---
## Task 3 — Data Visualization with Seaborn  *(4 marks)*

### Seaborn Plot 1 — Bar Plot: Math & Science by Pass/Fail

In [ ]:
# Add a readable 'Result' column for Seaborn grouping
df['Result'] = df['passed'].map({1: 'Pass', 0: 'Fail'})
palette = {'Pass': '#2ECC71', 'Fail': '#E74C3C'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Average Math & Science Score by Result', fontsize=14, fontweight='bold')

# Math subplot
sns.barplot(data=df, x='Result', y='math', hue='Result',
            palette=palette, legend=False, ax=ax1)
ax1.set_title('Math Score')
ax1.set_xlabel('Result')
ax1.set_ylabel('Average Score')

# Science subplot
sns.barplot(data=df, x='Result', y='science', hue='Result',
            palette=palette, legend=False, ax=ax2)
ax2.set_title('Science Score')
ax2.set_xlabel('Result')
ax2.set_ylabel('Average Score')

plt.tight_layout()
plt.savefig('plot6_seaborn_bar_math_science.png', dpi=120)
plt.show()
print('Saved: plot6_seaborn_bar_math_science.png')

### Seaborn Plot 2 — Scatter + Regression: Attendance vs Avg Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# One regplot per group (Pass and Fail)
sns.regplot(data=df[df['passed'] == 1], x='attendance_pct', y='avg_score',
            label='Pass',
            scatter_kws={'color': '#2ecc71', 's': 70},
            line_kws={'color': '#27ae60'},
            ax=ax)
sns.regplot(data=df[df['passed'] == 0], x='attendance_pct', y='avg_score',
            label='Fail',
            scatter_kws={'color': '#e74c3c', 's': 70, 'marker': 'X'},
            line_kws={'color': '#c0392b'},
            ax=ax)

ax.set_title('Attendance % vs Avg Score (with Regression Lines)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Attendance %')
ax.set_ylabel('Average Score')
ax.legend(title='Result')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot7_seaborn_regression.png', dpi=120)
plt.show()
print('Saved: plot7_seaborn_regression.png')

In [ ]:
# ─── Seaborn vs Matplotlib Comparison ─────────────────────────────────────
# Seaborn was noticeably easier for grouped statistical plots: sns.barplot()
# automatically computed means and confidence intervals without any manual
# aggregation. The hue parameter handled colour-grouping in a single call.
# Matplotlib required more explicit control — looping over groups, calling
# scatter() separately, and managing colours manually — but gave finer
# control over every visual element (tick labels, box colours, bar annotations).
print('Seaborn vs Matplotlib comparison noted as comment above.')

---
## Task 4 — Machine Learning with scikit-learn  *(8 marks + 2 bonus)*

### Step 1 — Prepare Data

In [ ]:
feature_cols = ['math', 'science', 'english', 'history', 'pe',
                'attendance_pct', 'study_hours_per_day']

# Separate features (X) and target (y)
# Note: 'name' and 'passed' are excluded from X
X = df[feature_cols]
y = df['passed']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale features — fit ONLY on training data, then transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {feature_cols}')

### Step 2 — Train a Logistic Regression Model

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
print(f'Training accuracy : {train_acc:.2%}')

### Step 3 — Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy : {test_acc:.2%}')

print('\nPer-student predictions on the test set:')
print(f"  {'Name':<10}  {'Actual':<7}  {'Predicted':<10}  {'Correct?'}")
print('  ' + '-' * 40)
for name, true, pred in zip(df.loc[y_test.index, 'name'], y_test, y_pred):
    label_true = 'Pass' if true == 1 else 'Fail'
    label_pred = 'Pass' if pred == 1 else 'Fail'
    correct    = '✓' if true == pred else '✗'
    print(f"  {correct} {name:<10}  {label_true:<7}  {label_pred:<10}")

print('\nNote: With only 3 test samples (15 × 20%), accuracy of 67% or 100%')
print('are both valid outcomes for such a small dataset.')

### Step 4 — Feature Importance (Coefficients)

In [ ]:
# Logistic Regression coefficients = log-odds weights per feature
coef_pairs = sorted(zip(feature_cols, model.coef_[0]),
                    key=lambda x: x[1])
features_sorted, coefs_sorted = zip(*coef_pairs)

print(f"{'Feature':<25} {'Coefficient':>12}  Direction")
print('-' * 58)
for feat, coef in coef_pairs:
    direction = '→ pushes toward Pass' if coef > 0 else '→ pushes toward Fail'
    print(f"{feat:<25} {coef:>+12.4f}  {direction}")

In [ ]:
# Horizontal bar chart coloured by sign
colors_coef = ['#e74c3c' if c < 0 else '#2ecc71' for c in coefs_sorted]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(features_sorted, coefs_sorted, color=colors_coef,
               edgecolor='white', linewidth=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(
    'Logistic Regression Feature Coefficients\n'
    '(green = pushes toward Pass, red = pushes toward Fail)',
    fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.set_ylabel('Feature')
for bar, val in zip(bars, coefs_sorted):
    ax.text(val + (0.015 if val >= 0 else -0.015),
            bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('plot8_feature_importance.png', dpi=120)
plt.show()
print('Saved: plot8_feature_importance.png')

### Step 5 — Predict for a New Student  *(Bonus: 2 marks)*

In [ ]:
# Define a new student [math, science, english, history, pe, attendance_pct, study_hours]
new_student = pd.DataFrame(
    [[75, 78, 66, 65, 80, 62, 3.2]],
    columns=feature_cols
)

# Scale using the already-fitted scaler
new_scaled = scaler.transform(new_student)

# Predict
prediction  = model.predict(new_scaled)[0]
probability = model.predict_proba(new_scaled)[0]
label       = 'Pass' if prediction == 1 else 'Fail'

print('=== New Student Prediction ===')
print(f'  Features    : {dict(zip(feature_cols, new_student.values[0]))}')
print(f'  Prediction  : {label}')
print(f'  P(Fail)     : {probability[0]:.2%}')
print(f'  P(Pass)     : {probability[1]:.2%}')

---
## Submission Checklist

| Item | Status |
|------|--------|
| `students.csv` committed to repo | ✅ |
| Task 1 — Pandas exploration (all 6 steps) | ✅ |
| Task 2 — 5 Matplotlib plots (saved as .png) | ✅ |
| Task 3 — 2 Seaborn plots (saved as .png) | ✅ |
| Task 3 — Seaborn vs Matplotlib comparison comment | ✅ |
| Task 4 — Logistic Regression (train, eval, coefficients) | ✅ |
| Task 4 Step 5 — New student prediction (bonus) | ✅ |
| Kernel → Restart & Run All executed before push | ⬜ |
| All inline outputs visible in committed notebook | ⬜ |